In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

ROOT           = Path.cwd().parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
MODELS         = ROOT / 'models'

df_original = pd.read_csv(ROOT / 'data' / 'raw' / 'Churn_Modelling.csv')
df_original = df_original.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

df_model = df_original.copy()
df_model = pd.get_dummies(df_model, columns=['Geography'], drop_first=True)
le = LabelEncoder()
df_model['Gender'] = le.fit_transform(df_model['Gender'])

X = df_model.drop(columns=['Exited'])
y = df_model['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled  = scaler.transform(X_test)

rf = joblib.load(MODELS / 'random_forest.pkl')

print("Everything loaded successfully")
print(f"df_original shape: {df_original.shape}")
print(f"df_model shape:    {df_model.shape}")
print(f"X shape:           {X.shape}")
print("Ready for churn risk scoring")

Everything loaded successfully
df_original shape: (10000, 11)
df_model shape:    (10000, 12)
X shape:           (10000, 11)
Ready for churn risk scoring


In [4]:
# Prepare full dataset for scoring
X_full = df_model.drop(columns=['Exited'])
X_full_scaled = scaler.transform(X_full)

# Get churn probability for every customer
churn_proba = rf.predict_proba(X_full_scaled)[:, 1]

# Add scores back to original dataframe
df_scored = df_original.copy()
df_scored['churn_probability'] = churn_proba.round(4)
df_scored['churn_prediction']  = (churn_proba >= 0.5).astype(int)

# Assign risk tier
def assign_tier(prob):
    if prob >= 0.75:  return 'High Risk'
    elif prob >= 0.5: return 'Medium Risk'
    elif prob >= 0.25: return 'Low Risk'
    else:             return 'Very Low Risk'

df_scored['risk_tier'] = df_scored['churn_probability'].apply(assign_tier)

print(df_scored['risk_tier'].value_counts())
print(df_scored.groupby('risk_tier')['churn_probability'].describe().round(3))

df_scored.to_csv('../data/processed/customers_scored.csv', index=False)
print('Scored customers saved')


risk_tier
Very Low Risk    5601
Low Risk         2053
Medium Risk      1204
High Risk        1142
Name: count, dtype: int64
                count   mean    std    min    25%    50%    75%    max
risk_tier                                                             
High Risk      1142.0  0.870  0.066  0.750  0.815  0.872  0.928  0.985
Low Risk       2053.0  0.360  0.072  0.250  0.296  0.353  0.418  0.500
Medium Risk    1204.0  0.618  0.071  0.500  0.555  0.615  0.678  0.750
Very Low Risk  5601.0  0.110  0.065  0.008  0.054  0.102  0.161  0.250
Scored customers saved


In [5]:
high_risk = df_scored[df_scored['risk_tier'] == 'High Risk']
all_customers = df_scored

print(f'High risk customers: {len(high_risk):,} ({len(high_risk)/len(all_customers)*100:.1f}%)')
print()

# Profile comparison
profile = pd.DataFrame({
    'All Customers':  all_customers[['Age','Balance','NumOfProducts',
                       'CreditScore','EstimatedSalary']].mean(),
    'High Risk':      high_risk[['Age','Balance','NumOfProducts',
                       'CreditScore','EstimatedSalary']].mean(),
}).round(2)

print('Profile comparison — High Risk vs All Customers:')
print(profile.to_string())

# Geography breakdown of high-risk
print('\nHigh-risk customers by Geography:')
print(high_risk['Geography'].value_counts())

# Gender breakdown
print('\nHigh-risk customers by Gender:')
print(high_risk['Gender'].value_counts())


High risk customers: 1,142 (11.4%)

Profile comparison — High Risk vs All Customers:
                 All Customers  High Risk
Age                      38.92      47.87
Balance               76485.89   94963.14
NumOfProducts             1.53       1.48
CreditScore             650.53     642.79
EstimatedSalary      100090.24  103708.34

High-risk customers by Geography:
Geography
Germany    632
France     324
Spain      186
Name: count, dtype: int64

High-risk customers by Gender:
Gender
Female    788
Male      354
Name: count, dtype: int64
